# Colored

In [18]:
def print_colored(*values,
                  color="red",
                  bg="white",
                  bold=True,
                  italic=False,
                  underline=False,
                  code=False,
                  sep=" ",
                  end="\n",
                  use_html="auto"):
                  
    """
    color="red", bg="white", bold=True
    Print colored/styled text in Jupyter notebooks (HTML) with ANSI fallback.

    Parameters:
      values:        items to print (like print())
      color:         foreground color name or hex (e.g. "red", "#ff8800")
      bg:            background color name or hex
      bold:          bool
      italic:        bool
      underline:     bool
      code:          bool (monospace + light background)
      sep, end:      like print()
      use_html:      "auto" | True | False
                    - "auto" uses HTML when running in IPython/Jupyter
    """
    text = sep.join(str(v) for v in values)

    # ---------- helpers ----------
    css_color_map = {
        "black": "#000000", "red": "#e53935", "green": "#43a047", "yellow": "#fdd835",
        "blue": "#1e88e5", "magenta": "#8e24aa", "cyan": "#00acc1", "white": "#ffffff",
        "gray": "#757575", "orange": "#fb8c00"
    }
    ansi_color_map = {
        "black": 30, "red": 31, "green": 32, "yellow": 33,
        "blue": 34, "magenta": 35, "cyan": 36, "white": 37,
        "gray": 90, "orange": 33  # orange isn't standard in ANSI; map to yellow-ish
    }
    def normalize_color(c, mapping):
        if not c:
            return None
        c_lower = str(c).lower().strip()
        return mapping.get(c_lower, c)  # allow hex or css strings for HTML

    # Detect IPython/Jupyter
    def in_ipython():
        try:
            from IPython import get_ipython
            return get_ipython() is not None
        except Exception:
            return False

    if use_html == "auto":
        use_html = in_ipython()

    # ---------- HTML path (best for notebooks) ----------
    if use_html:
        try:
            from IPython.display import display, HTML
            fg = normalize_color(color, css_color_map)
            bgc = normalize_color(bg, css_color_map)

            styles = []
            if fg: styles.append(f"color: {fg}")
            if bgc: styles.append(f"background-color: {bgc}")
            if bold: styles.append("font-weight: 700")
            if italic: styles.append("font-style: italic")
            if underline: styles.append("text-decoration: underline")

            if code:
                styles.append("font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace")
                styles.append("padding: 2px 6px")
                styles.append("border-radius: 6px")
                styles.append("background-color: rgba(0,0,0,0.06)" if not bg else "")
                styles.append("white-space: pre-wrap")

            style_str = "; ".join(s for s in styles if s)
            # Escape HTML special chars
            esc = (text.replace("&", "&amp;")
                        .replace("<", "&lt;")
                        .replace(">", "&gt;")
                        .replace('"', "&quot;"))
            html = f"<span style='{style_str}'>{esc}</span>"

            # Respect end: if end includes newline, add <br>
            if end:
                html += end.replace("\n", "<br>")
            display(HTML(html))
            return
        except Exception:
            # fall back to ANSI if HTML display fails
            pass

    # ---------- ANSI fallback (terminals / plain output) ----------
    ansi_parts = []
    if bold: ansi_parts.append("1")
    if italic: ansi_parts.append("3")
    if underline: ansi_parts.append("4")

    fg_code = normalize_color(color, ansi_color_map)
    bg_code = normalize_color(bg, ansi_color_map)

    if isinstance(fg_code, int):
        ansi_parts.append(str(fg_code))
    if isinstance(bg_code, int):
        ansi_parts.append(str(bg_code + 10))  # background is +10

    prefix = f"\033[{';'.join(ansi_parts)}m" if ansi_parts else ""
    suffix = "\033[0m" if ansi_parts else ""
    print(f"{prefix}{text}{suffix}", end=end)

# Collected Data

In [3]:
import pandas as pd
bq=pd.read_csv('data/bq-results-20260212.csv')
# Snippet
# website,activity,published,start_date,end_date,group_name,crawl_date,technology,categories,versions
# pindrophearing.co.uk,Healthcare,2024-08-21,2023-08-21,2025-08-21,apt73,2025-01-01,Priority Hints,Performance,""
# universalcompanies.com,Business Services,2024-09-14,2023-09-14,2025-09-14,play,2025-01-01,Priority Hints,Performance,""

# print num of rows
print(f"Number of rows: {len(bq)}")

Number of rows: 4330138


# Group By

In [19]:
# GROUP BY website,activity,published,start_date,end_date,group_name,crawl_date and create LIST of technologies
grouped = bq.groupby(['website','activity','published','start_date','end_date','group_name','crawl_date'])['technology'].apply(list).reset_index()
print(grouped.head())
# save to csv
grouped.to_csv('output/grouped_bq_results.csv', index=False)
print("Grouped data saved to output/grouped_bq_results.csv")

# save only where published month - crawl_date month is 3
grouped['published'] = pd.to_datetime(grouped['published'])
grouped['crawl_date'] = pd.to_datetime(grouped['crawl_date'])
grouped['month_diff'] = (grouped['published'].dt.year - grouped['crawl_date'].dt.year) * 12 + (grouped['published'].dt.month - grouped['crawl_date'].dt.month)
grouped_3_months = grouped[grouped['month_diff'] == 3]

#grouped_3_months.to_csv('output/grouped_bq_results_3_months.csv', index=False)
#print("Grouped data with 3 months difference saved to output/grouped_bq_results_3_months.csv")
# head
print("")
print_colored("Technologies of hacked site 3 Months before the attack:", color="blue", bold=True)
print(grouped_3_months.head())
grouped_3_months.to_csv('output/grouped_bq_results_3_months.csv', index=False)
print("Grouped data with 3 months difference saved to output/grouped_bq_results_3_months.csv")

                       website    activity   published  start_date    end_date  group_name  crawl_date                                         technology
0   http://www.informatika.com  Technology  2025-05-23  2024-05-23  2026-05-23  worldleaks  2024-06-01  [Bootstrap, Priority Hints, Google Maps, Micro...
1   http://www.informatika.com  Technology  2025-05-23  2024-05-23  2026-05-23  worldleaks  2024-07-01  [reCAPTCHA, Yoast SEO, LazySizes, Really Simpl...
2   http://www.informatika.com  Technology  2025-05-23  2024-05-23  2026-05-23  worldleaks  2024-08-01  [Yoast SEO, Swiper, Isotope, WordPress, core-j...
3   http://www.informatika.com  Technology  2025-05-23  2024-05-23  2026-05-23  worldleaks  2024-09-01  [MonsterInsights, RSS, jQuery, Really Simple S...
4   http://www.informatika.com  Technology  2025-05-23  2024-05-23  2026-05-23  worldleaks  2024-10-01  [WPML, MySQL, Google Analytics, Yoast SEO, Swi...
Grouped data saved to output/grouped_bq_results.csv



                                  website                         activity  published  start_date    end_date  group_name crawl_date                                         technology  month_diff
8              http://www.informatika.com                       Technology 2025-05-23  2024-05-23  2026-05-23  worldleaks 2025-02-01  [Google AdSense, WP Rocket, MonsterInsights, Y...           3
20             https://customfoodsinc.com                Consumer Services 2024-06-05  2023-06-05  2025-06-05       qilin 2024-03-01  [WordPress, Bootstrap, Yoast SEO, Isotope, Fon...           3
48                  https://www.dm.gov.ae                       Government 2024-06-05  2023-06-05  2025-06-05      daixin 2024-03-01  [Open Graph, Addsearch, jQuery UI, PHP, WordPr...           3
72   https://www.ganaderiarevuelta.com.mx  Agriculture and Food Production 2024-06-05  2023-06-05  2025-06-05       qilin 2024-03-01  [YouTube, Quantcast Measure, PHP, Google Font ...           3
96             https